## Technical Decision: Document Isolation & Physical Normalization Pipeline

**1. Context & Goal**
To ensure high-fidelity data extraction using Vision-Language Models (VLMs) and structural parsers (like Docling), we need to feed them perfectly prepared inputs. Passing a raw, multi-page, potentially poorly scanned PDF directly into these engines leads to three major issues:

* **Context Bloat:** Processing large documents at once exceeds token limits and causes "cross-page hallucinations."
* **Orientation Failures:** Technical documents often embed landscape tables (90°/270°) on portrait pages. Parsers strictly follow the PDF's `Rotation` metadata. If the metadata says 0° but the table is sideways, the extracted layout will be completely garbled.
* **Resolution Loss:** Default PDF rendering often drops the DPI, making small text in tables unreadable for Vision models.

**2. Architecture, Tooling & Licensing**
We designed a lightweight, CPU-friendly preprocessing pipeline that completely isolates and normalizes each page before downstream processing. All components use commercially permissive licenses, avoiding GPL/copyleft restrictions.

* **PDF Engine (`pypdfium2`):**
* *Why:* A Python binding for Google's PDFium (the engine behind Chrome). It is fast, uses C-level memory management, and handles both page extraction and high-res bitmap rendering without relying on heavy Java/system dependencies (unlike `pdf2image` + `poppler`).
* *License:* **Apache 2.0 / BSD-3-Clause**


* **Orientation AI (`onnxruntime` + `PP-LCNet_x1_0_doc_ori`):**
* *Why:* We use a layout-aware model (from PaddleX) compiled to ONNX. It natively detects 4 angles (0°, 90°, 180°, 270°) by looking at the page's global geometry (grids, margins). ONNX executes this ~6.5MB model in milliseconds on the CPU.
* *License:* **MIT** (ONNX Runtime) & **Apache 2.0** (PaddleOCR model)


* **Image Manipulation (`Pillow`):**
* *Why:* Used for memory-efficient pixel matrix rotation and saving the high-res renders for the VLM.
* *License:* **HPND** (Permissive, MIT-like)



**3. Pipeline Workflow (Step-by-Step)**

1. **Extraction:** The source PDF is split into single-page PDF files (`page_00XX.pdf`). This ensures complete structural isolation for Docling.
2. **High-Res Rendering:** The page is rendered into a bitmap at `scale=3` (approx. 216-300 DPI), creating a crystal-clear PNG for the VLM agent.
3. **Visual Inference:** The PNG is scaled to `224x224`, normalized (ImageNet stats), and passed to the ONNX model to detect the *true physical rotation* of the content.
4. **Physical Normalization (The Fix):** * If the page is crooked (e.g., 270°), we don't just tag it—we physically fix it.
* *PNG Rotation:* The pixel matrix is rotated using `Pillow`.
* *PDF Rotation:* The `/Rotate` metadata of the standalone PDF page is mathematically adjusted `(current_rot - detected_angle) % 360` so PDF viewers and parsers see it upright.


5. **State Management:** The orchestrator receives a list of dictionaries with perfectly aligned artifacts (`rotation` is always guaranteed to be `0` at output, while `original_rotation` is logged for debugging).

**4. Commercial Compliance & Result**
Because the entire stack relies on MIT and Apache 2.0 licenses, this ETL module can be safely embedded into proprietary, enterprise-grade AI applications without risking source-code disclosure.

**The Result:** Downstream components (VLM, Docling, RAG) receive single-page, high-resolution, perfectly upright files. They no longer need to worry about document orientation, coordinate transformations, or memory leaks from massive files.

## Links:

- Converted models:

    https://github.com/GreatV/oar-ocr/blob/main/docs/models.md


- The direct link to convert onnx `PP-LCNet_x1_0_doc_ori`:

    https://github.com/GreatV/oar-ocr/releases/download/v0.3.0/pp-lcnet_x1_0_doc_ori.onnx

- The original documentation:
    https://paddlepaddle.github.io/PaddleX/main/en/module_usage/tutorials/ocr_modules/doc_img_orientation_classification.html

## Test the pdf_processor.py

In [1]:
# Enable autoreload to automatically pick up changes in local modules
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
from pathlib import Path

from doc_agent.data.pdf_processor import slice_pdf_to_pages
from doc_agent.utils.logger import setup_logger

# Configure logging
logger = setup_logger(
    name="006_pdf_processor_test", 
    level=logging.INFO,
    log_file="notebook_experiments.log",
    log_dir=Path.cwd() / "logs"
)

# 1. Define paths for the real multi-page PDF, workspace, and the ONNX model
pdf_path = Path("data/01_raw/pue_1.1-1.3.pdf")  # Replace with your actual file
workspace = Path("data/02_interim/slicer_test_workspace")
model_path = Path("../models/page_orientation.onnx")

# 2. Execute ONLY the physical slicer component
logger.info(f"Starting physical slicing for: {pdf_path.name}...")

try:
    # Pass the model_path as the required third argument
    pages_data = slice_pdf_to_pages(pdf_path, workspace, model_path)
    
    # 3. Verify the output structure and the detected orientation
    logger.info(f"Success! Document sliced into {len(pages_data)} page(s).")

    for page in pages_data[:28]:
        logger.info(f"Page: {page['id']}")
        logger.info(f"PDF: {page['pdf'].relative_to(workspace.parent)}")
        logger.info(f"PNG: {page['png'].relative_to(workspace.parent)}")
        
        # Log the angle detected by the ONNX model
        logger.info(f"Detected Rotation: {page['rotation']}°") 
        logger.info("-" * 30)

    logger.info(f"Please inspect the '{workspace}' directory manually to verify the artifacts.")

except Exception as e:
    logger.error(f"Slicing failed: {e}", exc_info=True)

2026-05-11 20:53:30 |     INFO | 006_pdf_processor_test:3188454434.py:25 - Starting physical slicing for: pue_1.1-1.3.pdf...
2026-05-11 20:53:32 |     INFO | 006_pdf_processor_test:3188454434.py:32 - Success! Document sliced into 27 page(s).
2026-05-11 20:53:32 |     INFO | 006_pdf_processor_test:3188454434.py:35 - Page: page_0001
2026-05-11 20:53:32 |     INFO | 006_pdf_processor_test:3188454434.py:36 - PDF: slicer_test_workspace/01_pages_pdf/page_0001.pdf
2026-05-11 20:53:32 |     INFO | 006_pdf_processor_test:3188454434.py:37 - PNG: slicer_test_workspace/02_renders_png/page_0001_highres.png
2026-05-11 20:53:32 |     INFO | 006_pdf_processor_test:3188454434.py:40 - Detected Rotation: 0°
2026-05-11 20:53:32 |     INFO | 006_pdf_processor_test:3188454434.py:41 - ------------------------------
2026-05-11 20:53:32 |     INFO | 006_pdf_processor_test:3188454434.py:35 - Page: page_0002
2026-05-11 20:53:32 |     INFO | 006_pdf_processor_test:3188454434.py:36 - PDF: slicer_test_workspace/01_